In [1]:
import requests
import json
import re
import pandas as pd

In [2]:
def get_uniprot(accession):

    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"

    resp = requests.get(
        endpoint,
        headers={"Accept": "application/json"}
    )

    return resp

In [3]:
def uniprot_parse_response(resp):

    data = resp.json()
    accession = data["primaryAccession"]

    return {
        accession: {
            "organism": data["organism"]["scientificName"],
            "geneInfo": data.get("genes"),
            "sequenceInfo": data["sequence"],
            "type": "protein"
        }
    }

In [4]:
def get_ensembl(id):

    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"

    resp = requests.get(
        endpoint,
        headers={"Content-Type": "application/json"}
    )

    return resp

In [5]:
def ensembl_parse_response(resp):

    data = resp.json()
    ensembl_id = data["id"]

    return {
        ensembl_id: {
            "object_type": data.get("object_type"),
            "species": data.get("species"),
            "assembly_name": data.get("assembly_name"),
            "biotype": data.get("biotype"),
            "display_name": data.get("display_name"),
            "id": data.get("id"),
            "db_type": data.get("db_type"),
            "description": data.get("description"),
            "canonical_transcript": data.get("canonical_transcript"),
            "source": data.get("source")
        }
    }

In [6]:
def main(ids):

    results = {}

    for id in ids:

        try:

            if re.match(r"^[A-Z0-9]{6}$", id):

                resp = get_uniprot(id)

                if resp.status_code == 200:
                    parsed = uniprot_parse_response(resp)
                    results.update(parsed)
                else:
                    results[id] = "error:invalid uniprot id"

            elif re.match(r"^ENS[A-Z]*[0-9]+", id):

                resp = get_ensembl(id)

                if resp.status_code == 200:
                    parsed = ensembl_parse_response(resp)
                    results.update(parsed)
                else:
                    results[id] = "error:invalid ensembl id"

            else:
                results[id] = "error:unknown database"

        except Exception:
            results[id] = "error"

    return pd.DataFrame(results).T

In [7]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,object_type,species,assembly_name,biotype,display_name,id,db_type,description,canonical_transcript,source
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database,error:unknown database
ENSG00000157764,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRAF,ENSG00000157764,core,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRCA2,ENSG00000139618,core,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
